## Evolution Analysis

In [1]:
from datetime import datetime
date = datetime.now()
formatted_date = date.strftime("%B %d, %Y")
print(formatted_date)

June 02, 2025


In [2]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import userdata
userdata.get('HF_TOKEN')

# Set up the current working directory within the Google Drive
%cd /content/drive/My\ Drive/Colab\ Notebooks/LLM/sped_biblio/evolution

Mounted at /content/drive
/content/drive/My Drive/Colab Notebooks/LLM/sped_biblio/evolution


In [3]:
# !pip install dill qgrid

# !pip install spacy
# !python -m spacy download en_core_web_md
# !pip install --upgrade -q plotly
# !pip install -q pandas==2.2.2 numpy==1.26.4

In [4]:
import re
import ast
import warnings
from collections import defaultdict
import pickle
from pickle import UnpicklingError

# Data Manipulation
import numpy as np
import pandas as pd
import requests
import time
from collections import defaultdict

# Natural Language Processing
import requests
import re
import nltk
import spacy
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

# Network Analysis
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import networkx as nx

# Progress Bar
from tqdm import tqdm

# Display HTML
from IPython.display import IFrame

In [5]:
df = pd.read_excel("files/df.xlsx")

#### New Word Counts

In [6]:
def clean_literal(s):
    s = s.strip().replace('\n', ' ')
    if not s.startswith('['):
        s = '[' + s
    if not s.endswith(']'):
        s = s + ']'
    return s

def safe_eval_if_str(val, exclude_numbers=False):
    if isinstance(val, str):
        try:
            cleaned = clean_literal(val)
            tokens = ast.literal_eval(cleaned)
        except Exception:
            val = val.strip("[]")
            tokens = [item.strip(" '") for item in val.split(",")]
        if exclude_numbers:
            tokens = [token for token in tokens if not token.isdigit()]
        return tokens
    return val

In [7]:
df['unigrams'] = df['unigrams'].apply(safe_eval_if_str)
df['bigrams']  = df['bigrams'].apply(safe_eval_if_str)
df['trigrams'] = df['trigrams'].apply(safe_eval_if_str)

In [8]:
df_unigrams = df[['UT', 'WC', 'Year', 'Topic', 'unigrams']].copy()
df_unigrams['token_count'] = df_unigrams['unigrams'].apply(len)
df_unigrams = df_unigrams.explode('unigrams').rename(columns={'unigrams': 'ngram'})
df_unigrams['ngram_type'] = 'unigrams'

df_bigrams = df[['UT', 'WC', 'Year', 'Topic', 'bigrams']].copy()
df_bigrams['token_count'] = df_bigrams['bigrams'].apply(len)
df_bigrams = df_bigrams.explode('bigrams').rename(columns={'bigrams': 'ngram'})
df_bigrams['ngram_type'] = 'bigrams'

df_trigrams = df[['UT', 'WC', 'Year', 'Topic', 'trigrams']].copy()
df_trigrams['token_count'] = df_trigrams['trigrams'].apply(len)
df_trigrams = df_trigrams.explode('trigrams').rename(columns={'trigrams': 'ngram'})
df_trigrams['ngram_type'] = 'trigrams'

ngrams_concat = pd.concat([df_unigrams, df_bigrams, df_trigrams], ignore_index=True)

In [9]:
ngrams_concat.to_pickle("results/ngrams_concat.pkl")

In [10]:
unigrams_df = ngrams_concat[ngrams_concat['ngram_type'].str.lower() == 'unigrams'].copy()
unigrams_df['Decade'] = unigrams_df['Year'].apply(lambda y: f"{int(y)//10*10}s")

first_year = unigrams_df.groupby('ngram')['Year'].min().reset_index()
first_year['FirstDecade'] = first_year['Year'].apply(lambda y: f"{int(y)//10*10}s")
unigrams_df = unigrams_df.merge(first_year[['ngram', 'FirstDecade']], on='ngram', how='left')
unigrams_df.to_excel("results/unigrams_df.xlsx", index=False)

def process_tokens(tokens):
    cnt = tokens.value_counts()
    once = cnt[cnt == 1].index.tolist()
    multi = cnt[cnt > 1].index.tolist()
    all_distinct = cnt.index.tolist()

    freq_df = pd.DataFrame({'frequency': cnt})
    desc_stats = freq_df['frequency'].describe()
    skew_val = freq_df['frequency'].skew()
    kurt_val = freq_df['frequency'].kurt()

    return pd.DataFrame([{
        'Total Word Count': cnt.sum(),
        'Min': desc_stats['min'],
        '25%': desc_stats['25%'],
        '50%': desc_stats['50%'],
        '75%': desc_stats['75%'],
        'Max': desc_stats['max'],
        'Skew': round(skew_val, 2),
        'Kurt': round(kurt_val, 2),
        'Distinct Word Count': len(all_distinct),
        #'Distinct Words': all_distinct,
        'Count (Once)': len(once),
        #'Words Appearing Once': once,
        'Count (>1 Time)': len(multi),
        #'Words Appearing >1 Time': multi,
    }])

unigrams_group_df = []
for decade, sub_df in unigrams_df.groupby('Decade'):
    token_stats = process_tokens(sub_df['ngram'])
    current_decade_val = int(decade[:-1])

    reused_tokens_df = sub_df[sub_df['FirstDecade'].apply(lambda fd: int(fd[:-1]) < current_decade_val)]
    token_stats['Reused Word Count'] = reused_tokens_df['ngram'].nunique()
    reused_counts = reused_tokens_df['ngram'].value_counts()
    top_10_reused_tokens = reused_counts.head(10).index.tolist()
    token_stats['Top 10 Reused Words'] = ', '.join(top_10_reused_tokens)

    new_tokens_df = sub_df[sub_df['FirstDecade'] == decade]
    token_stats['New Word Count Per Decade'] = new_tokens_df['ngram'].nunique()
    new_counts = new_tokens_df['ngram'].value_counts()

    filtered_new_tokens = [tok for tok in new_counts.index.tolist() if not tok.isdigit()]
    top_10_new_tokens = filtered_new_tokens[:10]
    token_stats['Top 10 New Words'] = ', '.join(top_10_new_tokens)

    token_stats['Decade'] = decade
    unigrams_group_df.append(token_stats)

unigrams_concat = pd.concat(unigrams_group_df, ignore_index=True)
unigrams_concat = unigrams_concat.set_index('Decade').transpose()
unigrams_concat = unigrams_concat.reset_index().rename(columns={'index': 'Metric'})

def count_new_word_topic_year(df):
    grouped = df.groupby(['Topic', 'Year'])['ngram'].apply(set).reset_index()
    count_df = []
    for topic, group in grouped.groupby('Topic'):
        group = group.sort_values('Year')
        previous_tokens = set()
        for _, row in group.iterrows():
            current_tokens = row['ngram']
            new_tokens = current_tokens - previous_tokens
            count = len(new_tokens)
            count_df.append({
                'Topic': topic,
                'Year': row['Year'],
                'Count': count
            })
            previous_tokens.update(current_tokens)
    return pd.DataFrame(count_df)

new_word_count_df = count_new_word_topic_year(unigrams_df)
new_word_count_df['Decade'] = new_word_count_df['Year'].apply(lambda y: f"{int(y)//10*10}s")
new_word_topic_decade_count = new_word_count_df.groupby('Decade')['Count'].sum().reset_index()

unigrams_concat = unigrams_concat.set_index('Metric')
for _, row in new_word_topic_decade_count.iterrows():
    decade = row['Decade']
    count = row['Count']
    unigrams_concat.loc['New Word Per Topic Per Decade Count', decade] = count

unigrams_concat = unigrams_concat.reset_index()
# unigrams_concat

In [11]:
fig_unigrams = go.Figure(data=[go.Table(
    header=dict(
        values=list(unigrams_concat.columns),
        fill_color='#f2f2f2',
        align='center',
        font=dict(color='black', size=13, family='Arial'),
        line_color='darkslategray',
        height=40
    ),
    cells=dict(
        values=[unigrams_concat[col] for col in unigrams_concat.columns],
        fill_color=[['white', '#f9f9f9'] * (len(unigrams_concat) // 2 + 1)],
        align='center',
        font=dict(color='black', size=13, family='Arial'),
        line_color='lightgray',
        height=30
    )
)])

fig_unigrams.update_layout(
    width=1000,
    height=800,
    autosize=False,
    margin=dict(l=0, r=0, b=0, t=0)
)

fig_unigrams.write_html(
       "results/fig_unigrams.html",
       config={"responsive": True}
)

fig_unigrams.update_layout(width=None, height=None, autosize=True)
fig_unigrams.write_html(
    "results/fig_unigrams.html",
    config={"responsive": True}
)


In [17]:
IFrame(src='results/fig_unigrams.html', width=1000, height=800)

In [13]:
unigrams_concat.to_excel("results/unigrams_concat.xlsx", index=False)
new_word_count_df.to_excel("results/new_word_count_df.xlsx", index=False)

In [18]:
from nbconvert import HTMLExporter
import nbformat

notebook_path = 'index.ipynb'
html_exporter = HTMLExporter()

with open(notebook_path, 'r', encoding='utf-8') as nb_file:
    notebook_content = nb_file.read()
    notebook = nbformat.reads(notebook_content, as_version=4)

if 'widgets' in notebook.metadata and 'application/vnd.jupyter.widget-state+json' in notebook.metadata['widgets']:
    if 'state' not in notebook.metadata['widgets']['application/vnd.jupyter.widget-state+json']:
        notebook.metadata['widgets']['application/vnd.jupyter.widget-state+json']['state'] = {}

html_output, _ = html_exporter.from_notebook_node(notebook)

with open('index.html', 'w', encoding='utf-8') as html_file:
    html_file.write(html_output)